In [8]:
import torch
import subprocess
import os
import shutil
from gradio_client import Client, handle_file
import whisper
import shlex
import subprocess
import demucs.separate
from df.enhance import enhance, init_df, load_audio, save_audio
import torch
import gc
from transformers import AutoTokenizer
from gptqmodel import GPTQModel

def check_gpu():
    """Check if GPU is available with PyTorch."""
    print("Checking GPU availability with PyTorch...")
    if torch.cuda.is_available():
        print(f"GPU is available: {torch.cuda.get_device_name(0)}")
        return True
    else:
        print("GPU is not available.")
        return False

def check_whisper_installed():
    """Check if Whisper is installed."""
    print("Checking Whisper installation...")
    try:
        import whisper
        print("Whisper is installed.")
        return True
    except ImportError:
        print("Whisper is not installed.")
        return False

def check_ffmpeg():
    """Check if ffmpeg is installed."""
    print("Checking ffmpeg installation...")
    try:
        result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
        if "ffmpeg version" in result.stdout:
            print("ffmpeg is installed.")
            return True
        else:
            print("ffmpeg is not installed.")
            return False
    except FileNotFoundError:
        print("ffmpeg is not installed.")
        return False



/venv/lib/python3.10/site-packages/df/io.py:9: UserWarning: `torchaudio.backend.common.AudioMetaData` has been moved to `torchaudio.AudioMetaData`. Please update the import path.
  from torchaudio.backend.common import AudioMetaData


In [21]:
"""
Here we take the Raw audio(video) file and prepare the audio for the rest to come
"""
def separate_vocals(input_audio_path, output_dir, model='mdx_extra'):
    """
    Separate vocals and no-vocals with Demucs and save to custom output_dir.
    """
        
    print(f"Separating vocals from {input_audio_path}...")

    # Run Demucs CLI separation
    cmd = f'--two-stems vocals -n {model} "{input_audio_path}"'
    demucs.separate.main(shlex.split(cmd))

    base_name = os.path.splitext(os.path.basename(input_audio_path))[0]
    demucs_output_dir = os.path.join("separated", model, base_name)

    vocals_src = os.path.join(demucs_output_dir, "vocals.wav")
    no_vocals_src = os.path.join(demucs_output_dir, "no_vocals.wav")

    os.makedirs(output_dir, exist_ok=True)

    vocals_dst = os.path.join(output_dir, f"{base_name}_vocals.wav")
    no_vocals_dst = os.path.join(output_dir, f"{base_name}_no_vocals.wav")

    shutil.copy2(vocals_src, vocals_dst)
    shutil.copy2(no_vocals_src, no_vocals_dst)

    print(f"Vocals saved to: {vocals_dst}")
    print(f"No vocals saved to: {no_vocals_dst}")

    return vocals_dst, no_vocals_dst  # Return paths of saved files


def convert_to_48k_wav(input_audio_path, output_wav_path):
    """
    Convert any audio input to 48kHz stereo WAV using ffmpeg.
    """
    print(f"Converting {input_audio_path} to 48kHz WAV at {output_wav_path}...")
    subprocess.run([
        'ffmpeg', '-y', '-i', input_audio_path,
        '-ar', '48000',
        '-ac', '2',
        output_wav_path
    ], check=True)
    print("Conversion done.")


def denoise_audio(input_wav_path, output_wav_path):
    """
    Denoise/enhance the WAV audio using DeepFilterNet.
    """
    print(f"Denoising audio {input_wav_path} to {output_wav_path}...")
    model, df_state, _ = init_df()
    audio, _ = load_audio(input_wav_path, sr=df_state.sr())
    enhanced = enhance(model, df_state, audio)
    save_audio(output_wav_path, enhanced, df_state.sr())
    print("Denoising done.")


In [10]:
"""
here we take the prepared audio and make it into text
"""

def load_whisper_model(model_size="base"):
    """Load the Whisper model.
    
    Args:
        model_size: Size of the model to load (tiny, base, small, medium, large)
    
    Returns:
        The loaded Whisper model
    """
    print(f"Loading Whisper {model_size} model...")
    model = whisper.load_model(model_size)
    print(f"Whisper {model_size} model loaded successfully.")
    return model

def transcribe_audio(model, audio_file_path, language=None):
    """Transcribe audio using Whisper.
    
    Args:
        model: The loaded Whisper model
        audio_file_path: Path to the audio file to transcribe
        language: Optional language code for transcription
        
    Returns:
        Transcribed text
    """
    print(f"Transcribing audio file: {audio_file_path}")
    
    # Set transcription options
    options = {}
    if language:
        options["language"] = language
    
    # Perform transcription
    result = model.transcribe(audio_file_path, **options)
    
    transcribed_text = result["text"]
    print(f"Transcription complete. Transcribed text: {transcribed_text[:100]}...")
    return transcribed_text


def write_transcription_to_json(result, output_dir):
    """Write the full Whisper transcription result to a JSON file."""
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
        
    print(f"✅ JSON transcript saved to: {output_path}")
    
    
def save_transcribed_text(text, output_dir, filename_base):
    """Save transcribed text to a file.
    
    Args:
        text: Transcribed text to save
        output_dir: Directory to save the text file
        filename_base: Base filename (without extension)
        
    Returns:
        Path to the saved text file
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Create output file path
    output_file_path = os.path.join(output_dir, f"{filename_base}.txt")
    
    # Write text to file
    with open(output_file_path, 'w', encoding='utf-8') as f:
        f.write(text)
    
    print(f"Transcribed text saved to {output_file_path}")
    return output_file_path



In [11]:
def translate_transcribed_text(model_id, input_file_path, output_dir):
    """
    Translates transcribed text using a quantized LLM (e.g., GPTQ).
    
    Args:
        model_id: Hugging Face model ID (e.g., "MaziyarPanahi/Meta-Llama-3-8B-Instruct-GPTQ")
        input_file_path: Path to the transcribed .txt file
        output_dir: Directory to save the translated file
        
    Returns:
        Path to the saved translated file
    """
    # Clear memory
    try: del model
    except: pass
    try: del tokenizer
    except: pass
    torch.cuda.empty_cache()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()

    # Load model and tokenizer
    model = GPTQModel.from_quantized(
        model_id,
        device="cuda:0",
        trust_remote_code=True
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # Read original text
    with open(input_file_path, "r", encoding="utf-8") as file:
        original_text = file.read()

    # Prepare prompt
    messages = [
        {"role": "system", "content": "You are a translation chatbot that translates text from german to english! Only translate the given text."},
        {"role": "user", "content": original_text},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    # Generate response muessen tokens dann nochmal anpassen 
    outputs = model.generate(
        input_ids,
        do_sample=True,
        temperature=0.6,
        max_new_tokens=400,
        top_p=0.9,
    )

    # Decode and save result
    response = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
    translated_text = response.split("assistant")[0].strip()
    # Prepare output path
    base_filename = os.path.splitext(os.path.basename(input_file_path))[0]
    translated_filename = f"{base_filename}_translated.txt"
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, translated_filename)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(translated_text)

    print(f"Translated text saved to {output_path}")
    return output_path

In [12]:
def synthesize_speech(text, reference_audio_path, reference_text="", output_dir="../output"):
    """Synthesize speech using Fish Speech.
    
    Args:
        text: Text to synthesize
        reference_audio_path: Path to reference audio file
        reference_text: Optional reference text
        output_dir: Directory to save the output file
        
    Returns:
        Path to the generated audio file
    """
    print("Synthesizing speech with Fish Speech...")
    try:
        client = Client("http://fish-speech:7860/")
        
        result = client.predict(
            text=text,
            reference_id="",
            reference_audio=handle_file(reference_audio_path),
            reference_text=reference_text,
            max_new_tokens=0,
            chunk_length=200,
            top_p=0.5,
            repetition_penalty=1.7,
            temperature=0.6,
            seed=0,
            use_memory_cache="on",
            api_name="/partial"
        )
        
        # Handle the result
        if isinstance(result, tuple) and len(result) > 0:
            if isinstance(result[0], str) and result[0].endswith(".wav"):
                wav_file_path = result[0]
                
                # Create output directory if it doesn't exist
                os.makedirs(output_dir, exist_ok=True)
                
                # Save directly to output folder with original filename
                output_file_path = os.path.join(output_dir, os.path.basename(wav_file_path))
                shutil.move(wav_file_path, output_file_path)
                print(f"Generated .wav file saved to {output_file_path}")
                return output_file_path
            else:
                print(f"Unexpected result format: {result}")
                return None
        else:
            print(f"Unexpected result format: {result}")
            return None
    except Exception as e:
        print(f"Error in speech synthesis: {e}")
        return None


In [13]:
def process_audio_file(audio_file_path, reference_audio_path=None, whisper_model_size="base", language=None, output_dir="../output", model_id="MaziyarPanahi/Meta-Llama-3-8B-Instruct-GPTQ"):
    """Process an audio file: transcribe and then synthesize.
    
    Args:
        audio_file_path: Path to the audio file to transcribe
        reference_audio_path: Path to reference audio for synthesis (defaults to input audio if None)
        whisper_model_size: Size of the Whisper model to use
        language: Optional language code for transcription
        output_dir: Directory to save output files
    
    Returns:
        Tuple of (path to generated audio file, path to transcription file)
    """
    # Load Whisper model
    model = load_whisper_model(whisper_model_size)
    
    # Transcribe audio
    transcribed_text = transcribe_audio(model, audio_file_path, language)
    
    # Use the same audio file as reference if none provided
    if reference_audio_path is None:
        reference_audio_path = audio_file_path
    
    # Get base filename without extension for consistent naming
    base_filename = os.path.splitext(os.path.basename(audio_file_path))[0]
    
    # Save transcribed text to file
    text_file_path = save_transcribed_text(transcribed_text, output_dir, base_filename)

    # Translate described text and save text to file 
    translated_path = translate_transcribed_text(model_id, text_file_path, output_dir)
    
    # Synthesize speech using the transcribed text
    output_audio_path = synthesize_speech(transcribed_text, reference_audio_path, output_dir=output_dir)
    
    return output_audio_path, text_file_path, transcribed_text

In [22]:
    # Check requirements
    has_gpu = check_gpu()
    has_whisper = check_whisper_installed()
    has_ffmpeg = check_ffmpeg()
    
    if not (has_whisper and has_ffmpeg):
        print("Missing required components. Please install them before continuing.")
    
    # Process an audio file
    input_raw_audio = '../input/test_de.mp3'            # Your initial input audio file
    custom_folder = '../output/test-pipeline'     # Folder to save separated stems
    converted_wav = os.path.join(custom_folder, 'vocals_48k.wav')
    denoised_wav = os.path.join(custom_folder, 'vocals_denoised.wav')
    input_audio = "../input/test_de.mp3"
    reference_audio = "../input/test_de.mp3"  # Can be the same or different
    output_dir = "../output/test-pipeline"
    
    # Step 1: Separate vocals and no-vocals
    vocals_path, no_vocals_path = separate_vocals(input_audio, custom_folder)

    # Step 2: Convert vocals to 48kHz WAV
    convert_to_48k_wav(vocals_path, converted_wav)

    # Step 3: Denoise the converted WAV
    denoise_audio(converted_wav, denoised_wav)

    
    output_audio_path, text_file_path, transcribed_text = process_audio_file(
        audio_file_path=input_audio,
        reference_audio_path=reference_audio,
        whisper_model_size="base",
        language="",  # Set to the language of the audio or None for auto-detection
        output_dir=output_dir
    )
    
    if output_audio_path and text_file_path:
        print(f"Pipeline completed successfully!")
        print(f"Transcribed text: {transcribed_text[:100]}...")
        print(f"Generated audio saved to: {output_audio_path}")
        print(f"Transcription saved to: {text_file_path}")
    else:
        print("Pipeline failed to complete successfully.")

Checking GPU availability with PyTorch...
GPU is available: NVIDIA GeForce RTX 3070
Checking Whisper installation...
Whisper is installed.
Checking ffmpeg installation...
ffmpeg is installed.
Separating vocals from ../input/test_de.mp3...


UnboundLocalError: local variable 'model' referenced before assignment